# Train the Block Blast agent on Colab

Runtime → Change runtime type → **GPU**. Upload the project (zip of the repo without `.venv/`, `logs/`, `models/`) to Google Drive first, or clone it from your own git remote.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/Block Blast AI"  # adjust to where you put the repo
%cd "{PROJECT}"

Install the package. Colab's preinstalled CUDA torch is kept; `pip` ignores the uv CUDA index in `pyproject.toml`.

In [ ]:
!pip install -q -e .
!python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

Sanity checks: engine tests and env throughput.

In [ ]:
!pip install -q pytest
!python -m pytest tests/engine -q
!python scripts/benchmark_env.py --steps 20000

Train. Checkpoints and TensorBoard logs go to Drive, so a disconnect loses at most `train.checkpoint_freq` steps. Envs run in-process (one batched GPU forward per step), so 2 vCPUs are enough.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

In [ ]:
!python scripts/train.py train.n_envs=8 train.device=cuda train.total_timesteps=50000000 run_name=colab_ppo

Evaluate the best checkpoint against the greedy baseline on the held-out seeds.

In [ ]:
!python scripts/evaluate.py agent=greedy
!python scripts/evaluate.py agent=maskable_ppo eval.checkpoint=models/colab_ppo/best_model.zip eval.compare_to=data/eval/greedy_default.json